# Kaggle 50 Startups Analysis
## CRISP-DM + Scikit-Learn Machine Learning Project

**目標**: 建立可解釋的獲利預測模型

**資料集**: [Kaggle 50 Startups](https://www.kaggle.com/datasets/farhanmd29/50-startups)

---

### 商業問題
- 哪些因素最影響 Profit？
- R&D 是否比 Marketing 更重要？
- Administration 是否只是成本中心？
- State 是否影響 Profit？
- 如何最佳化資源配置？

## 1. Business Understanding

### 假設
- H1: R&D Spend 與 Profit 高度正相關
- H2: Marketing Spend 與 Profit 中度正相關
- H3: Administration 對 Profit 影響有限
- H4: State 對 Profit 影響不顯著

### 成功標準
- **商業**: 找出主要獲利因子、提供預算配置建議
- **技術**: R² ≥ 0.90

## 2. Data Understanding

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

df = pd.read_csv('../50_Startups.csv')
print(f'資料集形狀: {df.shape}')
df.head()

In [ ]:
# 資料型態
df.info()

In [ ]:
# 基本統計
df.describe()

In [ ]:
# 品質檢查
print(f'缺失值:\n{df.isnull().sum()}')
print(f'\n重複記錄: {df.duplicated().sum()}')

### 單變數分析

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, col in enumerate(['Profit', 'R&D Spend', 'Marketing Spend', 'Administration']):
    ax = axes[i // 2, i % 2]
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(f'{col} Distribution')
plt.tight_layout()
plt.show()

### 雙變數分析

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
pairs = [('R&D Spend', 'Profit'), ('Marketing Spend', 'Profit'), ('Administration', 'Profit')]
for i, (x, y) in enumerate(pairs):
    sns.scatterplot(data=df, x=x, y=y, ax=axes[i])
    axes[i].set_title(f'Profit vs {x}')
    corr = np.corrcoef(df[x], df[y])[0, 1]
    axes[i].text(0.05, 0.9, f'Corr: {corr:.3f}', transform=axes[i].transAxes,
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.tight_layout()
plt.show()

### 相關係數矩陣

In [ ]:
plt.figure(figsize=(8, 6))
corr = df[['R&D Spend', 'Administration', 'Marketing Spend', 'Profit']].corr()
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, fmt='.3f', square=True)
plt.title('Correlation Matrix')
plt.show()
print('\n與 Profit 相關性:')
print(corr['Profit'].drop('Profit').sort_values(ascending=False))

### State 分析 + ANOVA

In [ ]:
from scipy.stats import f_oneway

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='State', y='Profit')
plt.title('Profit Distribution by State')
plt.show()

groups = [g['Profit'].values for _, g in df.groupby('State')]
f_stat, p_val = f_oneway(*groups)
print(f'ANOVA: F={f_stat:.4f}, p-value={p_val:.6f}')
print(f'顯著性 (α=0.05): {"顯著" if p_val < 0.05 else "不顯著"}')

### 離群值分析

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for i, col in enumerate(['R&D Spend', 'Administration', 'Marketing Spend', 'Profit']):
    ax = axes[i // 2, i % 2]
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)][col]
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(f'{col} (Outliers: {len(outliers)})')
plt.tight_layout()
plt.show()

## 3. Data Preparation

In [ ]:
# Feature Engineering
df['TotalSpend'] = df['R&D Spend'] + df['Marketing Spend'] + df['Administration']
df['ROI'] = df['Profit'] / df['TotalSpend'].replace(0, np.nan)
df['RDRatio'] = df['R&D Spend'] / df['TotalSpend'].replace(0, np.nan)
df['MarketingRatio'] = df['Marketing Spend'] / df['TotalSpend'].replace(0, np.nan)

df[['TotalSpend', 'ROI', 'RDRatio', 'MarketingRatio']].head()

In [ ]:
# One-Hot Encoding
df_enc = pd.get_dummies(df, columns=['State'], drop_first=True)
df_enc.head()

In [ ]:
# Train/Test Split
from sklearn.model_selection import train_test_split

y = df_enc['Profit']
X = df_enc.drop(columns=['Profit'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f'Training: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Scaling
from sklearn.preprocessing import StandardScaler

scale_cols = ['R&D Spend', 'Administration', 'Marketing Spend']
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test_scaled[scale_cols] = scaler.transform(X_test[scale_cols])
X_train_scaled.head()

## 4. Multicollinearity Analysis (VIF)

In [ ]:
from sklearn.linear_model import LinearRegression

def compute_vif(df, features):
    vif_data = pd.DataFrame()
    vif_data['feature'] = features
    vif_data['VIF'] = [
        1.0 / (1.0 - LinearRegression().fit(
            df[[f for f in features if f != feat]], df[feat]
        ).score(df[[f for f in features if f != feat]], df[feat]))
        for feat in features
    ]
    return vif_data

vif_df = compute_vif(X_train_scaled.select_dtypes(include=[np.number]),
                     X_train_scaled.select_dtypes(include=[np.number]).columns.tolist())
vif_df

## 5. Feature Selection

In [ ]:
# LassoCV
from sklearn.linear_model import LassoCV

lasso = LassoCV(cv=5, random_state=42, max_iter=10000)
lasso.fit(X_train_scaled, y_train)

coef_df = pd.DataFrame({
    'feature': X_train_scaled.columns,
    'coefficient': lasso.coef_
})
coef_df['selected'] = np.abs(coef_df['coefficient']) > 1e-6
coef_df = coef_df.sort_values('coefficient', key=abs, ascending=False)
print(f'最佳 alpha: {lasso.alpha_:.6f}')
coef_df

In [ ]:
# RFE
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

rfe = RFE(LinearRegression(), n_features_to_select=max(1, X_train_scaled.shape[1] // 2))
rfe.fit(X_train_scaled, y_train)

rfe_df = pd.DataFrame({
    'feature': X_train_scaled.columns,
    'selected': rfe.support_,
    'rank': rfe.ranking_
}).sort_values('rank')
rfe_df

## 6. Modeling

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(random_state=42),
    'Lasso': Lasso(random_state=42),
    'RandomForest': RandomForestRegressor(n_estimators=200, random_state=42),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=200, random_state=42),
}

try:
    from xgboost import XGBRegressor
    models['XGBoost'] = XGBRegressor(n_estimators=200, random_state=42)
except ImportError:
    print('XGBoost not available')

try:
    from lightgbm import LGBMRegressor
    models['LightGBM'] = LGBMRegressor(n_estimators=200, random_state=42)
except ImportError:
    print('LightGBM not available')

trained = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    trained[name] = model
    print(f'  ✓ {name} 訓練完成')

## 7. Evaluation

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import cross_val_score

results = []
for name, model in trained.items():
    y_pred = model.predict(X_test_scaled)
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2')
    
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    results.append({
        'Model': name,
        'R²': round(r2, 4),
        'MAE': round(mae, 2),
        'RMSE': round(rmse, 2),
        'CV_R²': f'{cv_scores.mean():.4f}±{cv_scores.std():.4f}'
    })
    status = '✓' if r2 >= 0.90 else '✗'
    print(f'  {status} {name:20s} R²={r2:.4f} MAE={mae:.2f} RMSE={rmse:.2f}')

result_df = pd.DataFrame(results).sort_values('R²', ascending=False)
result_df

In [ ]:
# 最佳模型視覺化
best_name = result_df.iloc[0]['Model']
best_model = trained[best_name]

y_pred = best_model.predict(X_test_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred, alpha=0.7)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[0].set_xlabel('Actual Profit')
axes[0].set_ylabel('Predicted Profit')
axes[0].set_title(f'{best_name}: Actual vs Predicted')

# Residuals
residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.7)
axes[1].axhline(y=0, color='r', linestyle='--')
axes[1].set_xlabel('Predicted Profit')
axes[1].set_ylabel('Residuals')
axes[1].set_title(f'{best_name}: Residual Plot')

plt.tight_layout()
plt.show()
print(f'🏆 最佳模型: {best_name}')

## 8. Explainable AI

In [ ]:
# SHAP Analysis
try:
    import shap
    explainer = shap.Explainer(best_model, X_train_scaled)
    shap_values = explainer(X_test_scaled)
    
    shap.summary_plot(shap_values, X_test_scaled)
    plt.show()
    
    shap.plots.waterfall(shap_values[0])
    plt.show()
    
    mean_shap = np.abs(shap_values.values).mean(axis=0)
    importance_df = pd.DataFrame({
        'feature': X_test_scaled.columns,
        'mean_abs_shap': mean_shap
    }).sort_values('mean_abs_shap', ascending=False)
    print('SHAP 特徵重要度:')
    print(importance_df)
except Exception as e:
    print(f'SHAP 分析失敗 (非 tree 模型或 shap 未安裝): {e}')

In [ ]:
# Permutation Importance
from sklearn.inspection import permutation_importance

perm_result = permutation_importance(best_model, X_test_scaled, y_test, n_repeats=10, random_state=42)

perm_df = pd.DataFrame({
    'feature': X_test_scaled.columns,
    'importance': perm_result.importances_mean,
    'std': perm_result.importances_std
}).sort_values('importance', ascending=True)

plt.figure(figsize=(10, 5))
plt.barh(perm_df['feature'], perm_df['importance'], xerr=perm_df['std'])
plt.xlabel('Permutation Importance')
plt.title(f'Permutation Importance - {best_name}')
plt.tight_layout()
plt.show()
print(perm_df.sort_values('importance', ascending=False))

## 9. Business Analytics

In [ ]:
# What-If Analysis
base_row = X_test_scaled.iloc[0:1].copy()
base_pred = best_model.predict(base_row)[0]

scenarios = {
    'Increase_R&D_10%': ('R&D Spend', 1.10),
    'Increase_Marketing_10%': ('Marketing Spend', 1.10),
    'Decrease_Administration_10%': ('Administration', 0.90),
}

print(f'基準預測 Profit: {base_pred:.2f}')
for name, (col, factor) in scenarios.items():
    row = base_row.copy()
    if col in row.columns:
        row[col] = row[col] * factor
    pred = best_model.predict(row)[0]
    print(f'{name}: Profit={pred:.2f}, 變化={pred - base_pred:.2f}')

In [ ]:
# Budget Analysis: High vs Low Profit
df['TotalSpend'] = df['R&D Spend'] + df['Marketing Spend'] + df['Administration']
median_profit = df['Profit'].median()
high = df[df['Profit'] >= median_profit]
low = df[df['Profit'] < median_profit]

categories = ['R&D Spend', 'Marketing Spend', 'Administration']
comparison = pd.DataFrame({
    'Category': categories,
    'HighProfit_%': [high[c].mean() / high['TotalSpend'].mean() * 100 for c in categories],
    'LowProfit_%': [low[c].mean() / low['TotalSpend'].mean() * 100 for c in categories],
})
comparison

In [ ]:
# ROI Analysis
top_roi = df.nlargest(5, 'ROI')[['R&D Spend', 'Marketing Spend', 'Administration', 'Profit', 'ROI']]
top_roi

In [ ]:
# K-Means Clustering
from sklearn.cluster import KMeans

cluster_features = ['R&D Spend', 'Marketing Spend', 'Administration']
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(df[cluster_features])

cluster_labels = {0: 'InnovationDriven', 1: 'MarketingDriven', 2: 'Balanced'}
df['ClusterLabel'] = df['Cluster'].map(cluster_labels)

df.groupby('ClusterLabel')[cluster_features + ['Profit']].mean().round(2)

In [ ]:
# 集群視覺化
colors = {'InnovationDriven': 'red', 'MarketingDriven': 'blue', 'Balanced': 'green'}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label in cluster_labels.values():
    subset = df[df['ClusterLabel'] == label]
    axes[0].scatter(subset['R&D Spend'], subset['Profit'],
                    c=colors[label], label=label, alpha=0.7, s=80)
axes[0].set_xlabel('R&D Spend')
axes[0].set_ylabel('Profit')
axes[0].set_title('Clusters: R&D Spend vs Profit')
axes[0].legend()

for label in cluster_labels.values():
    subset = df[df['ClusterLabel'] == label]
    axes[1].scatter(subset['Marketing Spend'], subset['Profit'],
                    c=colors[label], label=label, alpha=0.7, s=80)
axes[1].set_xlabel('Marketing Spend')
axes[1].set_ylabel('Profit')
axes[1].set_title('Clusters: Marketing Spend vs Profit')
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. Business Insights & Recommendations

### 預期發現
1. **R&D 為最重要獲利因子** - 與 Profit 相關性最高，SHAP 重要度最高
2. **Marketing 為次要獲利因子** - 中度相關，有一定影響力
3. **Administration 影響有限** - 相關性低，非獲利驅動因素
4. **State 不一定具有統計顯著性** - ANOVA 結果不顯著
5. **高獲利企業具有特殊投資結構** - 集群分析顯示 Innovation Driven 類型獲利最高

### 建議
1. **增加 R&D 投資** - 最有效的獲利提升策略
2. **提高 Marketing 效率** - 優化行銷資源配置
3. **控制 Administration 成本** - 降低成本中心支出
4. **建立 Profit 預測模型** - 使用最佳模型進行預測
5. **建立預算配置決策系統** - Streamlit Dashboard 實現